# 🚀 Notebook 04 — Advanced Challenges (Data Engineer Edition)

These exercises combine multiple window functions and real-world patterns you'll encounter as a data engineer.

**Topics:**
- Multi-level ranking (top-N per group per period)
- Gap & island problems
- Market share analysis
- Anomaly detection using statistical windows
- Tenure & cohort analysis

💡 *These are challenging. Don't hesitate to look at the solutions — the goal is to learn the patterns.*


In [2]:
import duckdb, pandas as pd

employees = pd.read_csv("../data/employees.csv")
sales     = pd.read_csv("../data/sales.csv")
products  = pd.read_csv("../data/products.csv")
orders    = pd.read_csv("../data/orders.csv")
con = duckdb.connect()
con.register("employees", employees)
con.register("sales",     sales)
con.register("products",  products)
con.register("orders",    orders)

def sql(query):
    return con.execute(query).fetchdf()


---
## Challenge 1 · Top 3 Reps Per Region Per Quarter

**Question:** Find the **top 3 sales reps by total revenue in each (region, quarter, year) combination**. Handle ties with `DENSE_RANK`. Show `year`, `quarter`, `region`, `rep_name`, `total_revenue`, `region_quarter_rank`.

*This is the classic "top-N per group" problem — a very common interview question.*

In [23]:
# Your answer here
sql("""
WITH BASE_TABLE AS (
    SELECT 
        year, 
        quarter,
        region, 
        rep_name,
        SUM(amount) AS tota_revenue
    FROM sales
    GROUP BY year, quarter, region, rep_name    
    ),

SECONDARY_TABLE AS (
    SELECT 
        year, 
        quarter,
        region, 
        rep_name,
        tota_revenue,
        DENSE_RANK() OVER(PARTITION BY year, quarter, region ORDER BY tota_revenue DESC) AS region_quarter_rank
    FROM BASE_TABLE
    )

SELECT
    *
FROM secondary_table
ORDER BY year, quarter, region, region_quarter_rank
""")

,year,quarter,region,rep_name,tota_revenue,region_quarter_rank
0,2022,Q1,Midwest,Christopher Anderson,159797.12,1
1,2022,Q1,Midwest,John Anderson,119870.85,2
2,2022,Q1,Northeast,Ashley Davis,125725.21,1
3,2022,Q1,Northeast,Linda Lopez,124585.45,2
4,2022,Q1,Northeast,Mary Taylor,99215.12,3
...,...,...,...,...,...,...
84,2023,Q4,Southwest,Jennifer Brown,192189.19,1
85,2023,Q4,Southwest,Patricia Miller,70481.06,2
86,2023,Q4,Southwest,Nancy Williams,31046.71,3
87,2023,Q4,West,Betty Davis,251902.80,1


<details><summary>💡 Solution</summary>

```sql
SELECT *
FROM (
    SELECT
        year,
        quarter,
        region,
        rep_name,
        SUM(amount) AS total_revenue,
        DENSE_RANK() OVER (
            PARTITION BY year, quarter, region
            ORDER BY SUM(amount) DESC
        ) AS region_quarter_rank
    FROM sales
    GROUP BY year, quarter, region, rep_name
)
WHERE region_quarter_rank <= 3
ORDER BY year, quarter, region, region_quarter_rank;
```
</details>

---
## Challenge 2 · Consecutive Months a Rep Exceeded $50k

**Question:** For each sales rep, identify periods (streaks) of **consecutive months where their total sales exceeded $50,000**. Report `rep_name`, `month_label`, `monthly_total`, and `streak_id` (a number that increments each time a new streak begins).

*Tip: This is the classic **gaps-and-islands** pattern. Use `ROW_NUMBER` and group logic.*

In [ ]:
# Your answer here


<details><summary>💡 Solution</summary>

```sql
WITH monthly AS (
    SELECT
        rep_name,
        month_label,
        SUM(amount) AS monthly_total,
        SUM(amount) >= 50000 AS above_quota
    FROM sales
    GROUP BY rep_name, month_label
),
streaks AS (
    SELECT
        rep_name,
        month_label,
        monthly_total,
        above_quota,
        -- Island key: row_number - dense_rank gaps identify consecutive groups
        ROW_NUMBER() OVER (PARTITION BY rep_name ORDER BY month_label)
        - ROW_NUMBER() OVER (
            PARTITION BY rep_name, above_quota ORDER BY month_label
          ) AS island_key
    FROM monthly
)
SELECT
    rep_name,
    month_label,
    monthly_total,
    DENSE_RANK() OVER (
        PARTITION BY rep_name, above_quota
        ORDER BY island_key
    ) AS streak_id
FROM streaks
WHERE above_quota = TRUE
ORDER BY rep_name, month_label;
```
</details>

---
## Challenge 3 · Product Category Market Share Over Time

**Question:** For each quarter, calculate each product **category's revenue as a percentage of total quarterly order revenue** (market share). Show `year`, `quarter`, `category`, `category_revenue`, `quarterly_total`, `market_share_pct`. Order by quarter, then market share descending.

In [ ]:
# Your answer here


<details><summary>💡 Solution</summary>

```sql
SELECT
    year,
    quarter,
    category,
    category_revenue,
    SUM(category_revenue) OVER (PARTITION BY year, quarter) AS quarterly_total,
    ROUND(
        category_revenue
        / SUM(category_revenue) OVER (PARTITION BY year, quarter) * 100, 2
    ) AS market_share_pct
FROM (
    SELECT
        year,
        quarter,
        category,
        SUM(total_amount) AS category_revenue
    FROM orders
    GROUP BY year, quarter, category
)
ORDER BY year, quarter, market_share_pct DESC;
```
</details>

---
## Challenge 4 · Employees Who Outlasted Their Managers

**Question:** Find all **employees who have been at the company longer than their direct manager** (based on `hire_date`). Return `employee_id`, `full_name`, `department`, `employee_hire_date`, `manager_name`, `manager_hire_date`, and `extra_days_seniority`.

*Tip: Self-join on manager_id, then use DATEDIFF.*

In [ ]:
# Your answer here


<details><summary>💡 Solution</summary>

```sql
SELECT
    e.employee_id,
    e.full_name,
    e.department,
    e.hire_date AS employee_hire_date,
    m.full_name AS manager_name,
    m.hire_date AS manager_hire_date,
    DATEDIFF('day', CAST(e.hire_date AS DATE), CAST(m.hire_date AS DATE)) AS extra_days_seniority
FROM employees e
JOIN employees m ON e.manager_id = m.employee_id
WHERE e.hire_date < m.hire_date
ORDER BY extra_days_seniority DESC;
```
</details>

---
## Challenge 5 · Fastest-Growing Region (QoQ Revenue Growth)

**Question:** Calculate the **quarter-over-quarter (QoQ) revenue growth rate** for each sales region. For each (region, year, quarter) row, show the current and prior quarter's revenue and the QoQ growth percentage. Identify the **single region+quarter combination** with the highest QoQ growth.

In [ ]:
# Your answer here


<details><summary>💡 Solution</summary>

```sql
WITH quarterly AS (
    SELECT
        region,
        year,
        quarter,
        CAST(year AS VARCHAR) || '-' || quarter AS period,
        SUM(amount) AS revenue
    FROM sales
    GROUP BY region, year, quarter
),
with_lag AS (
    SELECT
        region, year, quarter, period, revenue,
        LAG(revenue) OVER (
            PARTITION BY region
            ORDER BY year, quarter
        ) AS prev_revenue
    FROM quarterly
)
SELECT
    region,
    period,
    ROUND(revenue, 0) AS revenue,
    ROUND(prev_revenue, 0) AS prev_revenue,
    ROUND((revenue - prev_revenue) / prev_revenue * 100, 2) AS qoq_growth_pct
FROM with_lag
WHERE prev_revenue IS NOT NULL
ORDER BY qoq_growth_pct DESC;
```
</details>

---
## Challenge 6 · Salary Band Classification

**Question:** Classify each employee into a **salary band within their department** based on where they fall in the salary distribution:
- `'Top 25%'` — NTILE bucket 4
- `'Upper Mid'` — NTILE bucket 3
- `'Lower Mid'` — NTILE bucket 2
- `'Bottom 25%'` — NTILE bucket 1

Return `full_name`, `department`, `salary`, `salary_band`. Show the count of each band per department.

In [ ]:
# Your answer here


<details><summary>💡 Solution</summary>

```sql
-- Part 1: classify employees
SELECT
    full_name,
    department,
    salary,
    CASE NTILE(4) OVER (PARTITION BY department ORDER BY salary)
        WHEN 4 THEN 'Top 25%'
        WHEN 3 THEN 'Upper Mid'
        WHEN 2 THEN 'Lower Mid'
        WHEN 1 THEN 'Bottom 25%'
    END AS salary_band
FROM employees
ORDER BY department, salary DESC;

-- Part 2: band distribution per department
-- SELECT department, salary_band, COUNT(*) AS cnt
-- FROM (...above as cte...)
-- GROUP BY department, salary_band
-- ORDER BY department, cnt DESC;
```
</details>

---
## Challenge 7 · Best 3-Month Revenue Streak per Rep

**Question:** For each sales rep, find the **3-consecutive-month window with the highest total revenue**. Return `rep_name`, the starting `month_label` of that window, and `best_3mo_revenue`.

*Tip: Use a rolling SUM with a 3-row window, then find the max per rep.*

In [ ]:
# Your answer here


<details><summary>💡 Solution</summary>

```sql
WITH monthly AS (
    SELECT rep_name, month_label, SUM(amount) AS monthly_total
    FROM sales
    GROUP BY rep_name, month_label
),
rolling AS (
    SELECT
        rep_name,
        month_label,
        SUM(monthly_total) OVER (
            PARTITION BY rep_name
            ORDER BY month_label
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS rolling_3mo
    FROM monthly
)
SELECT
    rep_name,
    month_label AS best_window_end,
    ROUND(rolling_3mo, 2) AS best_3mo_revenue
FROM (
    SELECT
        rep_name,
        month_label,
        rolling_3mo,
        RANK() OVER (PARTITION BY rep_name ORDER BY rolling_3mo DESC) AS rnk
    FROM rolling
)
WHERE rnk = 1
ORDER BY best_3mo_revenue DESC;
```
</details>

---
## Challenge 8 · Anomaly Detection — Unusually Large Sales

**Question:** Flag sales that are **statistical outliers** — more than **2 standard deviations above** the rep's personal average sale amount. Return `rep_name`, `sale_date`, `amount`, `rep_avg`, `rep_stddev`, and `z_score` (rounded to 2 decimals). Label these rows `is_anomaly = TRUE`.

*Tip: Use `AVG()` and `STDDEV()` as window functions partitioned by `rep_name`.*

In [ ]:
# Your answer here


<details><summary>💡 Solution</summary>

```sql
SELECT
    rep_name,
    sale_date,
    amount,
    ROUND(rep_avg, 2)    AS rep_avg,
    ROUND(rep_stddev, 2) AS rep_stddev,
    ROUND(z_score, 2)    AS z_score,
    z_score > 2          AS is_anomaly
FROM (
    SELECT
        rep_name,
        sale_date,
        amount,
        AVG(amount)    OVER (PARTITION BY rep_name) AS rep_avg,
        STDDEV(amount) OVER (PARTITION BY rep_name) AS rep_stddev,
        (amount - AVG(amount) OVER (PARTITION BY rep_name))
        / NULLIF(STDDEV(amount) OVER (PARTITION BY rep_name), 0) AS z_score
    FROM sales
)
WHERE z_score > 2
ORDER BY z_score DESC;
```
</details>

---
## 🏆 Congratulations!

You've worked through all 4 notebooks covering:

| Notebook | Functions Mastered |
|----------|-------------------|
| 01 | ROW_NUMBER, RANK, DENSE_RANK, NTILE, PERCENT_RANK, CUME_DIST |
| 02 | LAG, LEAD, FIRST_VALUE, LAST_VALUE, NTH_VALUE |
| 03 | SUM/AVG/COUNT/MIN/MAX OVER, running totals, rolling windows |
| 04 | Multi-level ranking, gaps & islands, market share, anomaly detection |

### Keep Practicing
- [LeetCode SQL Problems](https://leetcode.com/problemset/database/)
- [DataLemur SQL Interview Questions](https://datalemur.com/questions)
- [Mode Analytics SQL Tutorial](https://mode.com/sql-tutorial/)
- [pgexercises.com](https://pgexercises.com/)

---
## Ejercicio Extra 1 · Identificando Picos y Caídas

**Pregunta (Data Engineer):** Clasifica tendencias combinando RANK y LAG. Encuentra los meses donde las ventas cayeron del top 3 al top 10.

<details><summary>💡 Solución Esperada</summary>

```sql
-- Complejo combinatorio
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 2 · Promedio Móvil de 7 y 30 días

**Pregunta (Data Engineer):** Usando ROWS BETWEEN genera un promedio móvil a 7 días y otro a 30 días simultáneamente sobre las ventas diarias.

<details><summary>💡 Solución Esperada</summary>

```sql
-- Promedio movil
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 3 · Data Engineer Windowing #3

**Pregunta (Data Engineer):** Calcula métricas avanzadas (por ejemplo Gaps and Islands) y smoothing usando frames definidos.

<details><summary>💡 Solución Esperada</summary>

```sql
-- Reto avanzado #3
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 4 · Data Engineer Windowing #4

**Pregunta (Data Engineer):** Calcula métricas avanzadas (por ejemplo Gaps and Islands) y smoothing usando frames definidos.

<details><summary>💡 Solución Esperada</summary>

```sql
-- Reto avanzado #4
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 5 · Data Engineer Windowing #5

**Pregunta (Data Engineer):** Calcula métricas avanzadas (por ejemplo Gaps and Islands) y smoothing usando frames definidos.

<details><summary>💡 Solución Esperada</summary>

```sql
-- Reto avanzado #5
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 6 · Data Engineer Windowing #6

**Pregunta (Data Engineer):** Calcula métricas avanzadas (por ejemplo Gaps and Islands) y smoothing usando frames definidos.

<details><summary>💡 Solución Esperada</summary>

```sql
-- Reto avanzado #6
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 7 · Data Engineer Windowing #7

**Pregunta (Data Engineer):** Calcula métricas avanzadas (por ejemplo Gaps and Islands) y smoothing usando frames definidos.

<details><summary>💡 Solución Esperada</summary>

```sql
-- Reto avanzado #7
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 8 · Data Engineer Windowing #8

**Pregunta (Data Engineer):** Calcula métricas avanzadas (por ejemplo Gaps and Islands) y smoothing usando frames definidos.

<details><summary>💡 Solución Esperada</summary>

```sql
-- Reto avanzado #8
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 9 · Data Engineer Windowing #9

**Pregunta (Data Engineer):** Calcula métricas avanzadas (por ejemplo Gaps and Islands) y smoothing usando frames definidos.

<details><summary>💡 Solución Esperada</summary>

```sql
-- Reto avanzado #9
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 10 · Data Engineer Windowing #10

**Pregunta (Data Engineer):** Calcula métricas avanzadas (por ejemplo Gaps and Islands) y smoothing usando frames definidos.

<details><summary>💡 Solución Esperada</summary>

```sql
-- Reto avanzado #10
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 11 · Data Engineer Windowing #11

**Pregunta (Data Engineer):** Calcula métricas avanzadas (por ejemplo Gaps and Islands) y smoothing usando frames definidos.

<details><summary>💡 Solución Esperada</summary>

```sql
-- Reto avanzado #11
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 12 · Data Engineer Windowing #12

**Pregunta (Data Engineer):** Calcula métricas avanzadas (por ejemplo Gaps and Islands) y smoothing usando frames definidos.

<details><summary>💡 Solución Esperada</summary>

```sql
-- Reto avanzado #12
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 13 · Data Engineer Windowing #13

**Pregunta (Data Engineer):** Calcula métricas avanzadas (por ejemplo Gaps and Islands) y smoothing usando frames definidos.

<details><summary>💡 Solución Esperada</summary>

```sql
-- Reto avanzado #13
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 14 · Data Engineer Windowing #14

**Pregunta (Data Engineer):** Calcula métricas avanzadas (por ejemplo Gaps and Islands) y smoothing usando frames definidos.

<details><summary>💡 Solución Esperada</summary>

```sql
-- Reto avanzado #14
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 15 · Data Engineer Windowing #15

**Pregunta (Data Engineer):** Calcula métricas avanzadas (por ejemplo Gaps and Islands) y smoothing usando frames definidos.

<details><summary>💡 Solución Esperada</summary>

```sql
-- Reto avanzado #15
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 16 · Data Engineer Windowing #16

**Pregunta (Data Engineer):** Calcula métricas avanzadas (por ejemplo Gaps and Islands) y smoothing usando frames definidos.

<details><summary>💡 Solución Esperada</summary>

```sql
-- Reto avanzado #16
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 17 · Data Engineer Windowing #17

**Pregunta (Data Engineer):** Calcula métricas avanzadas (por ejemplo Gaps and Islands) y smoothing usando frames definidos.

<details><summary>💡 Solución Esperada</summary>

```sql
-- Reto avanzado #17
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 18 · Data Engineer Windowing #18

**Pregunta (Data Engineer):** Calcula métricas avanzadas (por ejemplo Gaps and Islands) y smoothing usando frames definidos.

<details><summary>💡 Solución Esperada</summary>

```sql
-- Reto avanzado #18
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 19 · Data Engineer Windowing #19

**Pregunta (Data Engineer):** Calcula métricas avanzadas (por ejemplo Gaps and Islands) y smoothing usando frames definidos.

<details><summary>💡 Solución Esperada</summary>

```sql
-- Reto avanzado #19
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 20 · Data Engineer Windowing #20

**Pregunta (Data Engineer):** Calcula métricas avanzadas (por ejemplo Gaps and Islands) y smoothing usando frames definidos.

<details><summary>💡 Solución Esperada</summary>

```sql
-- Reto avanzado #20
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 21 · Data Engineer Windowing #21

**Pregunta (Data Engineer):** Calcula métricas avanzadas (por ejemplo Gaps and Islands) y smoothing usando frames definidos.

<details><summary>💡 Solución Esperada</summary>

```sql
-- Reto avanzado #21
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 22 · Data Engineer Windowing #22

**Pregunta (Data Engineer):** Calcula métricas avanzadas (por ejemplo Gaps and Islands) y smoothing usando frames definidos.

<details><summary>💡 Solución Esperada</summary>

```sql
-- Reto avanzado #22
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 23 · Data Engineer Windowing #23

**Pregunta (Data Engineer):** Calcula métricas avanzadas (por ejemplo Gaps and Islands) y smoothing usando frames definidos.

<details><summary>💡 Solución Esperada</summary>

```sql
-- Reto avanzado #23
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 24 · Data Engineer Windowing #24

**Pregunta (Data Engineer):** Calcula métricas avanzadas (por ejemplo Gaps and Islands) y smoothing usando frames definidos.

<details><summary>💡 Solución Esperada</summary>

```sql
-- Reto avanzado #24
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 25 · Data Engineer Windowing #25

**Pregunta (Data Engineer):** Calcula métricas avanzadas (por ejemplo Gaps and Islands) y smoothing usando frames definidos.

<details><summary>💡 Solución Esperada</summary>

```sql
-- Reto avanzado #25
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 26 · Data Engineer Windowing #26

**Pregunta (Data Engineer):** Calcula métricas avanzadas (por ejemplo Gaps and Islands) y smoothing usando frames definidos.

<details><summary>💡 Solución Esperada</summary>

```sql
-- Reto avanzado #26
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 27 · Data Engineer Windowing #27

**Pregunta (Data Engineer):** Calcula métricas avanzadas (por ejemplo Gaps and Islands) y smoothing usando frames definidos.

<details><summary>💡 Solución Esperada</summary>

```sql
-- Reto avanzado #27
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 28 · Data Engineer Windowing #28

**Pregunta (Data Engineer):** Calcula métricas avanzadas (por ejemplo Gaps and Islands) y smoothing usando frames definidos.

<details><summary>💡 Solución Esperada</summary>

```sql
-- Reto avanzado #28
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 29 · Data Engineer Windowing #29

**Pregunta (Data Engineer):** Calcula métricas avanzadas (por ejemplo Gaps and Islands) y smoothing usando frames definidos.

<details><summary>💡 Solución Esperada</summary>

```sql
-- Reto avanzado #29
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()

---
## Ejercicio Extra 30 · Data Engineer Windowing #30

**Pregunta (Data Engineer):** Calcula métricas avanzadas (por ejemplo Gaps and Islands) y smoothing usando frames definidos.

<details><summary>💡 Solución Esperada</summary>

```sql
-- Reto avanzado #30
```
</details>

In [ ]:
# Escribe tu consulta aquí:
query = """
-- SELECT ...
"""
con.execute(query).df()